# Paper 2 final evidence: main results
This notebook summarizes the reproduced temporal and LOSO evidence and writes the F1/T8 support tables and figure. It does not assemble the paper.


## 1. Locate the evidence bundle
The next cell resolves the repository and run directory so the notebook can execute from any working directory inside the checkout.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "data" / "splits").is_dir() and (parent / "notebooks").is_dir()
)
EVIDENCE_DIR = PROJECT_ROOT / "notebooks/experiment/paper2-final-evidence-1.0"
COMPARATOR_DIR = EVIDENCE_DIR / "comparators"
FIGURES_DIR = EVIDENCE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Evidence directory: {EVIDENCE_DIR}")
print(f"Formal-eval comparator CSV: {(COMPARATOR_DIR / 'formal_eval_1.0_temporal_seed_summary.csv').exists()}")

Evidence directory: /scratch/group/p.cis250607.000/MDR-Project/notebooks/experiment/paper2-final-evidence-1.0
Formal-eval comparator CSV: True


## 2. Check the comparator rows
The formal-eval rows provide sensitivity-anchored historical comparisons; they are labeled separately from the paired Guarded-versus-Backbone reproduction.

In [2]:
formal_path = COMPARATOR_DIR / "formal_eval_1.0_temporal_seed_summary.csv"
formal_seed = pd.read_csv(formal_path)
required_sensitivity = {
    "Clustering_V0_Full_k2", "Global_Single", "Trained_Gating_k2",
    "Clustering_Dynamic_k2", "Seasonal_Binary_k2", "Univariate_G_API_k2",
}
formal_no_delta = formal_seed[
    formal_seed["strategy_name"].isin(required_sensitivity)
    & ((formal_seed["config_id"].str.endswith("_c0_0_c1_0"))
       | formal_seed["config_id"].eq("Global_Single_54"))
].copy()
counts = formal_no_delta.groupby("strategy_name")["seed"].nunique()
assert required_sensitivity.issubset(set(counts.index)), counts.to_dict()
assert bool((counts.loc[list(required_sensitivity)] == 30).all()), counts.to_dict()
print(counts.sort_index().to_string())


strategy_name
Clustering_Dynamic_k2    30
Clustering_V0_Full_k2    30
Global_Single            30
Seasonal_Binary_k2       30
Trained_Gating_k2        30
Univariate_G_API_k2      30


## 3. Build the temporal comparison table
The Guarded and unguarded Backbone rows come from the paired reproduction; V0, global, and other router rows are sensitivity-anchored to formal-eval 1.0.

In [3]:
from scipy.stats import t as student_t

routing_temporal = pd.read_csv(EVIDENCE_DIR / "temporal_seed_summary.csv")
expected_temporal_names = {
    "Guarded_Backbone54_k2", "Clustering_Backbone54_k2",
    "StationMean_Backbone54_k2",
}
expected_temporal_seeds = {
    42, 7, 13, 21, 55, 99, 123, 2024, 3141, 2718, 808, 1618, 2357,
    3333, 4321, 5555, 6789, 7777, 8888, 9091, 101, 202, 303, 404,
    505, 606, 707, 909, 1111, 2222,
}
assert len(routing_temporal) == 90
assert set(routing_temporal["strategy_name"]) == expected_temporal_names
for strategy, group in routing_temporal.groupby("strategy_name"):
    assert len(group) == 30, (strategy, len(group))
    assert set(group["seed"].astype(int)) == expected_temporal_seeds
paired_names = {"Guarded_Backbone54_k2", "Clustering_Backbone54_k2"}
paired_temporal = routing_temporal[routing_temporal["strategy_name"].isin(paired_names)].copy()
paired_counts = paired_temporal.groupby("strategy_name")["seed"].nunique()
assert set(paired_counts.index) == paired_names
assert bool((paired_counts == 30).all()), paired_counts.to_dict()

formal_rows = formal_no_delta.copy()
formal_rows["evidence_role"] = "sensitivity_anchored"
formal_rows["source_harness"] = "derived_8.4-formal-eval-1.0"
paired_temporal["evidence_role"] = "paired_in_harness"
paired_temporal["source_harness"] = "paper2-final-evidence-1.0"
comparison_seed = pd.concat([formal_rows, paired_temporal], ignore_index=True, sort=False)
comparison_seed.to_csv(EVIDENCE_DIR / "main_temporal_seed_comparison.csv", index=False)

name_map = {
    "Guarded_Backbone54_k2": "Guarded backbone (primary)",
    "Clustering_Backbone54_k2": "Backbone (paired ablation)",
    "Clustering_V0_Full_k2": "V0 (sensitivity)",
    "Global_Single": "Global (sensitivity)",
    "Trained_Gating_k2": "Target-derived gate (sensitivity)",
    "Clustering_Dynamic_k2": "Dynamic covariate (sensitivity)",
    "Seasonal_Binary_k2": "Seasonal (sensitivity)",
    "Univariate_G_API_k2": "G_API (sensitivity)",
}
summary_rows = []
for (strategy, role, harness), group in comparison_seed.groupby(
        ["strategy_name", "evidence_role", "source_harness"], sort=False):
    values = group["r2"].dropna().astype(float)
    n = int(values.size)
    mean = float(values.mean())
    sd = float(values.std(ddof=1))
    half = float(student_t.ppf(0.975, n - 1) * sd / np.sqrt(n))
    summary_rows.append({
        "strategy_name": strategy, "display_name": name_map[strategy],
        "evidence_role": role, "source_harness": harness, "n_seeds": n,
        "mean_r2": mean, "seed_sd_r2": sd, "seed_ci95_low": mean - half,
        "seed_ci95_high": mean + half, "mean_rmse": float(group["rmse"].mean()),
        "interval_scope": "expert seed variation only",
    })
main_temporal_summary = pd.DataFrame(summary_rows).sort_values("mean_rmse")
main_temporal_summary.to_csv(EVIDENCE_DIR / "main_temporal_summary.csv", index=False)
print(f"Reproduced routing rows={len(routing_temporal)} across {len(expected_temporal_names)} pinned configurations")
print(main_temporal_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


Reproduced routing rows=90 across 3 pinned configurations
           strategy_name                      display_name        evidence_role              source_harness  n_seeds  mean_r2  seed_sd_r2  seed_ci95_low  seed_ci95_high  mean_rmse             interval_scope
   Clustering_V0_Full_k2                  V0 (sensitivity) sensitivity_anchored derived_8.4-formal-eval-1.0       30 0.811843    0.001369       0.811332        0.812354   0.044187 expert seed variation only
   Guarded_Backbone54_k2        Guarded backbone (primary)    paired_in_harness   paper2-final-evidence-1.0       30 0.811843    0.001369       0.811332        0.812354   0.044187 expert seed variation only
Clustering_Backbone54_k2        Backbone (paired ablation)    paired_in_harness   paper2-final-evidence-1.0       30 0.811724    0.001369       0.811213        0.812235   0.044201 expert seed variation only
   Clustering_Dynamic_k2   Dynamic covariate (sensitivity) sensitivity_anchored derived_8.4-formal-eval-1.0       

## 4. Recompute the paired T8 LOSO differences
The next cell pairs Guarded and Backbone by seed and held-out station, then exports the per-fold values and station-level summary.

In [4]:
routing_loso = pd.read_csv(EVIDENCE_DIR / "loso_seed_station.csv")
expected_loso_names = {
    "Guarded_Backbone54_k2", "Clustering_Backbone54_k2",
    "StationMean_Backbone54_k2",
}
expected_loso_seeds = {42, 7, 13, 21, 55}
expected_loso_stations = {
    "BeaverPass_WA_990", "CayusePass_WA", "Darrington", "Paradise_WA",
    "Quinault", "SourdoughGulch_WA_985", "Spokane",
}
assert len(routing_loso) == 105
assert set(routing_loso["strategy_name"]) == expected_loso_names
assert set(routing_loso["seed"].astype(int)) == expected_loso_seeds
assert set(routing_loso["station"]) == expected_loso_stations
for strategy, group in routing_loso.groupby("strategy_name"):
    assert len(group) == 35, (strategy, len(group))
    assert set(group["seed"].astype(int)) == expected_loso_seeds
    assert set(group["station"]) == expected_loso_stations

guarded_loso = routing_loso.query("strategy_name == 'Guarded_Backbone54_k2'")[
    ["seed", "station", "r2"]
].rename(columns={"r2": "guarded_r2"})
backbone_loso = routing_loso.query("strategy_name == 'Clustering_Backbone54_k2'")[
    ["seed", "station", "r2"]
].rename(columns={"r2": "backbone_r2"})
t8_seed = guarded_loso.merge(backbone_loso, on=["seed", "station"], validate="one_to_one")
t8_seed["r2_diff_guarded_minus_backbone"] = t8_seed["guarded_r2"] - t8_seed["backbone_r2"]
assert len(t8_seed) == 35 and t8_seed["station"].nunique() == 7

def wins(values):
    return int((values > 1e-9).sum())

def ties(values):
    return int((values.abs() <= 1e-9).sum())

t8_station = t8_seed.groupby("station", as_index=False).agg(
    guarded_mean_r2=("guarded_r2", "mean"), backbone_mean_r2=("backbone_r2", "mean"),
    mean_r2_difference=("r2_diff_guarded_minus_backbone", "mean"),
    seed_sd_difference=("r2_diff_guarded_minus_backbone", "std"),
    winning_seeds=("r2_diff_guarded_minus_backbone", wins),
    tied_seeds=("r2_diff_guarded_minus_backbone", ties))
t8_seed.to_csv(EVIDENCE_DIR / "t8_loso_paired_differences.csv", index=False)
t8_station.to_csv(EVIDENCE_DIR / "t8_loso_station_summary.csv", index=False)
print(f"Reproduced LOSO rows={len(routing_loso)} across {len(expected_loso_names)} pinned configurations")
print(t8_station.sort_values("mean_r2_difference", ascending=False).to_string(
    index=False, float_format=lambda value: f"{value:.6f}"))


Reproduced LOSO rows=105 across 3 pinned configurations
              station  guarded_mean_r2  backbone_mean_r2  mean_r2_difference  seed_sd_difference  winning_seeds  tied_seeds
SourdoughGulch_WA_985         0.369860          0.310754            0.059106            0.008469              5           0
              Spokane         0.608266          0.556819            0.051448            0.004587              5           0
          Paradise_WA         0.779974          0.764531            0.015444            0.009670              5           0
           Darrington         0.694881          0.694881            0.000000            0.000000              0           5
        CayusePass_WA         0.686169          0.686169            0.000000            0.000000              0           5
    BeaverPass_WA_990         0.753115          0.753115            0.000000            0.000000              0           5
             Quinault         0.572875          0.572875            0.000000

## 5. Plot F1 temporal results and T8 station differences
The figure reads only the exported seed-summary and paired LOSO tables, keeping sensitivity-anchored and same-harness comparisons visibly distinct.

In [5]:
main_temporal_summary = pd.read_csv(EVIDENCE_DIR / "main_temporal_summary.csv")
t8_station = pd.read_csv(EVIDENCE_DIR / "t8_loso_station_summary.csv")

role_colors = {
    "paired_in_harness": "#1b9e77",
    "sensitivity_anchored": "#7570b3",
}
plot_temporal = main_temporal_summary.sort_values("mean_r2", ascending=True).reset_index(drop=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
y = np.arange(len(plot_temporal))
left = plot_temporal["mean_r2"] - plot_temporal["seed_ci95_low"]
right = plot_temporal["seed_ci95_high"] - plot_temporal["mean_r2"]
for role, rows in plot_temporal.groupby("evidence_role", sort=False):
    idx = rows.index.to_numpy()
    ax.errorbar(rows["mean_r2"], idx,
                xerr=np.vstack([left.iloc[idx], right.iloc[idx]]),
                fmt="o", markersize=6, capsize=3, linewidth=1.3,
                color=role_colors[role], label=role.replace("_", " "))
ax.set_yticks(y, plot_temporal["display_name"])
ax.set_xlabel("Temporal R²")
ax.set_title("F1 · Temporal performance")
ax.grid(axis="x", alpha=0.25)
ax.legend(fontsize=8, loc="lower right")

ax = axes[1]
t8_plot = t8_station.sort_values("mean_r2_difference", ascending=True).reset_index(drop=True)
bar_colors = np.where(t8_plot["mean_r2_difference"] >= 0, "#1b9e77", "#d95f02")
ax.barh(np.arange(len(t8_plot)), t8_plot["mean_r2_difference"],
        xerr=t8_plot["seed_sd_difference"], color=bar_colors,
        alpha=0.9, capsize=3)
ax.axvline(0.0, color="black", linewidth=0.9)
ax.set_yticks(np.arange(len(t8_plot)), t8_plot["station"])
ax.set_xlabel("Guarded − Backbone mean R²")
ax.set_title("T8 · Paired LOSO difference by station")
ax.grid(axis="x", alpha=0.25)
for y_pos, row in enumerate(t8_plot.itertuples()):
    positive = row.mean_r2_difference > 1e-9
    ax.text(row.mean_r2_difference - 0.002 if positive else 0.001, y_pos,
            f"{row.winning_seeds}/{row.tied_seeds}",
            va="center", ha="right" if positive else "left",
            color="white" if positive else "black", fontsize=8)
ax.set_xlim(-0.002, 0.071)

fig.suptitle("Routing reproduction · intervals reflect expert-seed variation", fontsize=14)
fig.text(0.01, 0.005,
         "F1 bars show seed-level 95% t intervals, not sampling or selection uncertainty. "
         "T8 labels show winning/tied seeds; error bars show SD across five paired seeds.",
         fontsize=8)
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
main_figure_path = FIGURES_DIR / "F1_T8_main_evidence.png"
fig.savefig(main_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print("F1 temporal summary:")
print(main_temporal_summary[["display_name", "evidence_role", "n_seeds", "mean_r2",
                             "seed_ci95_low", "seed_ci95_high"]].to_string(
    index=False, float_format=lambda value: f"{value:.6f}"))
print("T8 paired station table:")
print(t8_station.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print(f"Saved figure: {main_figure_path}")


F1 temporal summary:
                     display_name        evidence_role  n_seeds  mean_r2  seed_ci95_low  seed_ci95_high
                 V0 (sensitivity) sensitivity_anchored       30 0.811843       0.811332        0.812354
       Guarded backbone (primary)    paired_in_harness       30 0.811843       0.811332        0.812354
       Backbone (paired ablation)    paired_in_harness       30 0.811724       0.811213        0.812235
  Dynamic covariate (sensitivity) sensitivity_anchored       30 0.785464       0.785093        0.785836
             Global (sensitivity) sensitivity_anchored       30 0.779794       0.779316        0.780272
           Seasonal (sensitivity) sensitivity_anchored       30 0.770020       0.769438        0.770602
              G_API (sensitivity) sensitivity_anchored       30 0.767566       0.767216        0.767917
Target-derived gate (sensitivity) sensitivity_anchored       30 0.735359       0.734950        0.735768
T8 paired station table:
              stat